# 03. Model Development & Student Profile Segmentation

## Student Social Profile Segmentation Using Clustering

Notebook ini membandingkan clustering pada 36 fitur minat hasil preprocessing, memilih model dengan mempertimbangkan metrik dan ukuran segmen, lalu memprofilkan hasilnya menggunakan fitur minat mentah dan metadata profil. Seluruh proses yang mengandung randomness memakai `random_state=42`.

## 1. Setup dan Pemuatan Data

Memuat representasi aktif `log1p`-scaled sebagai input utama, representasi biner untuk validasi bentuk data, matriks minat mentah untuk profiling, serta metadata profil bersih. Path dibangun relatif terhadap folder notebook sehingga tetap benar bila kernel dimulai dari root project atau folder `notebooks/`.

In [ ]:
import os
os.environ["LOKY_MAX_CPU_COUNT"] = str(max(1, (os.cpu_count() or 1) - 1))

from pathlib import Path
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)

RANDOM_STATE = 42
sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams["figure.titlesize"] = 14
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.mixture")

CURRENT_DIR = Path.cwd()
NOTEBOOK_DIR = CURRENT_DIR if CURRENT_DIR.name == "notebooks" else CURRENT_DIR / "notebooks"
PROJECT_ROOT = NOTEBOOK_DIR.parent
PREPROCESSING_DIR = PROJECT_ROOT / "outputs" / "preprocessing"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
MODEL_DIR = PROJECT_ROOT / "outputs" / "models"
for directory in (FIGURE_DIR, TABLE_DIR, MODEL_DIR):
    directory.mkdir(parents=True, exist_ok=True)

X_active_scaled = pd.read_csv(
    PREPROCESSING_DIR / "X_active_log1p_scaled.csv", index_col=0
)
X_active_binary = pd.read_csv(
    PREPROCESSING_DIR / "X_active_binary.csv", index_col=0
)
X_interest_raw = pd.read_csv(
    PREPROCESSING_DIR / "X_interest_raw.csv", index_col=0
)
profiles = pd.read_csv(
    PREPROCESSING_DIR / "student_profiles_clean.csv", index_col=0
)

FEATURE_COLS = X_active_scaled.columns.tolist()
assert len(FEATURE_COLS) == 36
assert X_active_scaled.index.equals(X_active_binary.index)
assert X_active_scaled.columns.equals(X_active_binary.columns)
assert X_active_scaled.index.isin(X_interest_raw.index).all()
assert profiles.index.equals(X_interest_raw.index)
assert X_active_scaled.notna().all().all()

X_active_raw = X_interest_raw.loc[X_active_scaled.index, FEATURE_COLS]
print(f"Data aktif untuk clustering: {X_active_scaled.shape[0]:,} baris x {len(FEATURE_COLS)} fitur")
print(f"Data biner validasi: {X_active_binary.shape}; profil lengkap: {profiles.shape}")
print("Fitur clustering:", FEATURE_COLS)

**Interpretasi:** Hanya 36 fitur minat yang masuk ke ruang clustering. Atribut `gradyear`, `gender`, `age`, dan `NumberOffriends` disimpan untuk profiling setelah label terbentuk. Data biner dan data mentah disejajarkan melalui indeks profil; indeks CSV bukan fitur model.

## 2. Penentuan Jumlah Klaster K-Means

Elbow dan Silhouette dievaluasi untuk `k=2..10` pada kedua representasi (log1p-scaled dan binary) dengan sampel aktif acak tetap hingga 5.000 profil. Sampling membatasi biaya komputasi, terutama perhitungan Silhouette yang tumbuh kuadratik. Pilihan `k` untuk tiap representasi mengutamakan Silhouette dengan syarat setiap klaster mencakup setidaknya 2% sampel bila ada kandidat yang memenuhi.

In [ ]:
K_SAMPLE_SIZE = min(5000, len(X_active_scaled))
X_k_selection = X_active_scaled.sample(
    n=K_SAMPLE_SIZE, random_state=RANDOM_STATE
).sort_index()

k_records = []
for k in range(2, 11):
    candidate = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = candidate.fit_predict(X_k_selection)
    counts = pd.Series(labels).value_counts(normalize=True)
    binary_candidate = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    binary_labels = binary_candidate.fit_predict(
        X_active_binary.loc[X_k_selection.index]
    )
    binary_counts = pd.Series(binary_labels).value_counts(normalize=True)
    k_records.append({
        "k": k,
        "inertia": candidate.inertia_,
        "silhouette": silhouette_score(
            X_k_selection, labels, sample_size=min(2000, len(X_k_selection)),
            random_state=RANDOM_STATE
        ),
        "smallest_cluster_share": counts.min(),
        "binary_inertia": binary_candidate.inertia_,
        "binary_silhouette": silhouette_score(
            X_active_binary.loc[X_k_selection.index], binary_labels,
            sample_size=min(2000, len(X_k_selection)), random_state=RANDOM_STATE
        ),
        "binary_smallest_cluster_share": binary_counts.min(),
    })

k_selection = pd.DataFrame(k_records)
eligible_k = k_selection[k_selection["smallest_cluster_share"] >= 0.02]
selection_pool = eligible_k if not eligible_k.empty else k_selection
selected_k = int(
    selection_pool.sort_values(
        ["silhouette", "smallest_cluster_share"], ascending=[False, False]
    ).iloc[0]["k"]
)
eligible_binary_k = k_selection[k_selection["binary_smallest_cluster_share"] >= 0.02]
binary_selection_pool = eligible_binary_k if not eligible_binary_k.empty else k_selection
selected_k_binary = int(
    binary_selection_pool.sort_values(
        ["binary_silhouette", "binary_smallest_cluster_share"],
        ascending=[False, False],
    ).iloc[0]["k"]
)
k_selection.to_csv(TABLE_DIR / "k_selection_metrics.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(k_selection["k"], k_selection["inertia"], marker="o", label="log1p-scaled")
axes[0].plot(k_selection["k"], k_selection["binary_inertia"], marker="o", label="binary")
axes[0].axvline(selected_k, color="tab:red", linestyle="--", label=f"Selected k={selected_k}")
axes[0].axvline(selected_k_binary, color="tab:green", linestyle=":", label=f"Binary k={selected_k_binary}")
axes[0].set(title="Elbow Method", xlabel="Jumlah klaster (k)", ylabel="Inertia")
axes[0].legend()
axes[1].plot(k_selection["k"], k_selection["silhouette"], marker="o", label="log1p-scaled")
axes[1].plot(k_selection["k"], k_selection["binary_silhouette"], marker="o", label="binary")
axes[1].axvline(selected_k, color="tab:red", linestyle="--", label=f"Selected k={selected_k}")
axes[1].axvline(selected_k_binary, color="tab:green", linestyle=":", label=f"Binary k={selected_k_binary}")
axes[1].set(title="Silhouette Score", xlabel="Jumlah klaster (k)", ylabel="Silhouette")
axes[1].legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "09_kmeans_k_selection.png", dpi=150, bbox_inches="tight")
plt.show()
display(k_selection.round(4))
print(f"k terpilih log1p-scaled: {selected_k}; k terpilih binary: {selected_k_binary}.")
print("Pemilihan: Silhouette tertinggi, dengan preferensi ukuran minimum klaster >= 2%.")

**Interpretasi:** Bandingkan kurva kedua representasi bersama-sama. Inertia selalu turun ketika `k` bertambah, sehingga elbow bukan bukti tunggal. `k` dipilih per representasi melalui Silhouette tertinggi dengan preferensi ukuran minimum klaster >=2%; keputusan akhir tetap mempertimbangkan interpretabilitas dan keseimbangan segmen.

## 3. Pembandingan Algoritma dan Diagnostik Jarak

K-Means pada kedua representasi, Ward Agglomerative, DBSCAN, dan diagonal-covariance GMM dibandingkan pada sampel aktif deterministik. Agglomerative dan DBSCAN bersifat eksploratif; model final dipilih dari K-Means/GMM karena keduanya dapat menghasilkan assignment untuk seluruh profil aktif. Dendrogram dihitung pada subset 1.000 titik untuk menjaga biaya kuadratik tetap terkendali. `eps` DBSCAN ditaksir dari titik siku kurva jarak ke tetangga ke-10; noise dilaporkan terpisah.

In [ ]:
COMPARISON_SAMPLE_SIZE = min(2500, len(X_active_scaled))
X_model = X_active_scaled.sample(
    n=COMPARISON_SAMPLE_SIZE, random_state=RANDOM_STATE
).sort_index()

kmeans_sample = KMeans(
    n_clusters=selected_k, n_init=10, random_state=RANDOM_STATE
)
labels_kmeans = kmeans_sample.fit_predict(X_model)
X_binary_model = X_active_binary.loc[X_model.index]
kmeans_binary_sample = KMeans(
    n_clusters=selected_k_binary, n_init=10, random_state=RANDOM_STATE
)
labels_kmeans_binary = kmeans_binary_sample.fit_predict(X_binary_model)

agglomerative = AgglomerativeClustering(n_clusters=selected_k, linkage="ward")
labels_agglomerative = agglomerative.fit_predict(X_model)

MIN_SAMPLES = 10
neighbors = NearestNeighbors(n_neighbors=MIN_SAMPLES, n_jobs=1)
neighbor_distances, _ = neighbors.fit(X_model).kneighbors(X_model)
k_distances = np.sort(neighbor_distances[:, -1])
positive_k_distances = k_distances[k_distances > 1e-4]
if positive_k_distances.size == 0:
    raise ValueError("Tidak ada jarak k-neighbor positif untuk menentukan eps DBSCAN.")
normalized_x = np.linspace(0, 1, len(positive_k_distances))
distance_range = np.ptp(positive_k_distances)
normalized_y = (positive_k_distances - positive_k_distances.min()) / distance_range
knee_index = int(np.argmax(normalized_y - normalized_x))
EPS_ESTIMATE = float(positive_k_distances[knee_index])
# eps diambil dari knee lalu dibulatkan ke atas 2 desimal. Nilai knee jatuh tepat pada jarak antar titik,
# sehingga kalau dipakai apa adanya hasil DBSCAN bisa berbeda antar komputer.
EPS_USED = float(np.ceil(EPS_ESTIMATE * 100) / 100)
dbscan = DBSCAN(eps=EPS_USED, min_samples=MIN_SAMPLES, n_jobs=1)
labels_dbscan = dbscan.fit_predict(X_model)

gmm = GaussianMixture(
    n_components=selected_k, covariance_type="diag", n_init=1,
    max_iter=150, reg_covar=1e-4, random_state=RANDOM_STATE
)
labels_gmm = gmm.fit_predict(X_model)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(k_distances, color="tab:blue")
axes[0].axhline(EPS_ESTIMATE, color="tab:red", linestyle="--", label=f"eps={EPS_ESTIMATE:.3f}")
axes[0].set(title=f"{MIN_SAMPLES}-neighbor k-distance", xlabel="Profil (jarak diurutkan)", ylabel="Jarak")
axes[0].legend()
dendrogram_sample = X_model.sample(
    n=min(1000, len(X_model)), random_state=RANDOM_STATE
).sort_index()
hierarchy = linkage(dendrogram_sample, method="ward")
dendrogram(hierarchy, ax=axes[1], truncate_mode="lastp", p=35, show_leaf_counts=True)
axes[1].set(title="Dendrogram Ward (subset deterministik)", xlabel="Kelompok daun", ylabel="Jarak")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "10_density_and_dendrogram.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Sampel pembanding: {len(X_model):,}; k={selected_k}; DBSCAN eps={EPS_USED:.2f} (knee {EPS_ESTIMATE:.4f}), min_samples={MIN_SAMPLES}")
print("Label DBSCAN:", pd.Series(labels_dbscan).value_counts().sort_index().to_dict())

**Interpretasi:** Dendrogram hanya merangkum struktur subset, bukan seluruh 13.762 profil. `eps` DBSCAN adalah estimasi kurva jarak dan hasilnya perlu dinilai dari jumlah klaster, persentase noise, serta metrik pada inlier. DBSCAN yang menghasilkan satu klaster atau semua noise tidak memiliki Silhouette yang valid.

## 4. Evaluasi dan Pemilihan Model

Metrik dihitung pada hasil fit di sampel pembanding: Silhouette dan Calinski–Harabasz lebih tinggi umumnya lebih baik, Davies–Bouldin lebih rendah lebih baik. Baris DBSCAN dievaluasi hanya pada inlier dan menyertakan coverage/noise agar keterbatasan tersebut terlihat. Untuk pemilihan model yang akan memberi label ke seluruh data, K-Means dan GMM dirangking dari ketiga metrik; Agglomerative dan DBSCAN tetap menjadi pembanding eksploratif.

In [ ]:
def clustering_metrics(X, labels):
    labels = np.asarray(labels)
    unique_labels = np.unique(labels)
    if len(unique_labels) < 2 or len(unique_labels) >= len(labels):
        return np.nan, np.nan, np.nan
    return (
        silhouette_score(X, labels),
        davies_bouldin_score(X, labels),
        calinski_harabasz_score(X, labels),
    )

comparison_specs = [
    ("K-Means (log1p-scaled)", labels_kmeans, np.ones(len(labels_kmeans), dtype=bool)),
    ("K-Means (binary)", labels_kmeans_binary, np.ones(len(labels_kmeans_binary), dtype=bool)),
    ("Agglomerative Ward", labels_agglomerative, np.ones(len(labels_agglomerative), dtype=bool)),
    ("DBSCAN", labels_dbscan, labels_dbscan != -1),
    ("Gaussian Mixture (diag, log1p-scaled)", labels_gmm, np.ones(len(labels_gmm), dtype=bool)),
]
comparison_records = []
for model_name, labels, metric_mask in comparison_specs:
    metric_labels = labels[metric_mask]
    metric_X = X_model.loc[metric_mask]
    silhouette, db_index, ch_index = clustering_metrics(metric_X, metric_labels)
    cluster_sizes_sample = pd.Series(labels[labels != -1]).value_counts(normalize=True)
    comparison_records.append({
        "model": model_name,
        "role": "Baseline" if model_name == "K-Means (log1p-scaled)" else "Kandidat",
        "clusters_excluding_noise": len(np.unique(labels[labels != -1])),
        "noise_share": float(np.mean(labels == -1)),
        "smallest_cluster_share": float(cluster_sizes_sample.min()) if len(cluster_sizes_sample) else np.nan,
        "silhouette": silhouette,
        "davies_bouldin": db_index,
        "calinski_harabasz": ch_index,
        "metric_profiles": int(metric_mask.sum()),
        "metric_scope": "inliers only" if model_name == "DBSCAN" else "full comparison sample",
    })

model_comparison = pd.DataFrame(comparison_records)
deployable = model_comparison[model_comparison["model"].isin([
    "K-Means (log1p-scaled)", "K-Means (binary)",
    "Gaussian Mixture (diag, log1p-scaled)",
])].copy()
for metric, ascending in [
    ("silhouette", False),
    ("davies_bouldin", True),
    ("calinski_harabasz", False),
]:
    deployable[f"{metric}_rank"] = deployable[metric].rank(ascending=ascending, method="average")
deployable["mean_metric_rank"] = deployable[
    ["silhouette_rank", "davies_bouldin_rank", "calinski_harabasz_rank"]
].mean(axis=1)
deployable = deployable.sort_values(
    ["mean_metric_rank", "silhouette", "smallest_cluster_share"],
    ascending=[True, False, False],
)
best_algorithm = deployable.iloc[0]["model"]
best_feature_representation = (
    "binary" if best_algorithm == "K-Means (binary)" else "log1p_scaled"
)
model_comparison.to_csv(TABLE_DIR / "model_comparison.csv", index=False)
display(model_comparison.round(4))
print("Model assignment penuh terpilih:", best_algorithm)
print("Pemilihan memakai rerata peringkat tiga metrik pada K-Means/GMM yang bisa memberi label penuh.")

**Interpretasi:** Pilihan akhir adalah model dengan rerata peringkat terbaik di antara model yang dapat menetapkan label untuk data baru. Kualitas metrik tidak cukup bila segmen terlalu kecil atau tidak dapat dijelaskan; periksa ukuran segmen dan profil pada tahap berikutnya. DBSCAN dinilai hanya pada inlier sehingga nilainya tidak sebanding langsung dengan model lain ketika noise besar.

## 5–7. Fit Penuh, Visualisasi, Profiling, dan Penyimpanan

Model terpilih di-fit ulang pada seluruh profil aktif. Profil tanpa minat tercatat diberi `cluster_id=0` sebagai segmen deskriptif terpisah; label model aktif digeser menjadi 1..k agar tidak berbenturan. PCA dipakai hanya untuk visualisasi 2D—bukan sebagai input model. Statistik profil menggunakan hitungan minat mentah agar mudah ditafsirkan.

In [ ]:
if best_algorithm == "K-Means (log1p-scaled)":
    best_model = KMeans(
        n_clusters=selected_k, n_init=10, random_state=RANDOM_STATE
    ).fit(X_active_scaled)
    active_labels = best_model.labels_
    best_cluster_count = selected_k
elif best_algorithm == "K-Means (binary)":
    best_model = KMeans(
        n_clusters=selected_k_binary, n_init=10, random_state=RANDOM_STATE
    ).fit(X_active_binary)
    active_labels = best_model.labels_
    best_cluster_count = selected_k_binary
else:
    best_model = GaussianMixture(
        n_components=selected_k, covariance_type="diag", n_init=1,
        max_iter=150, reg_covar=1e-4, random_state=RANDOM_STATE
    ).fit(X_active_scaled)
    active_labels = best_model.predict(X_active_scaled)
    best_cluster_count = selected_k

clustered_data = profiles.join(X_interest_raw)
clustered_data["cluster_id"] = 0
clustered_data.loc[X_active_scaled.index, "cluster_id"] = active_labels + 1
clustered_data["cluster_id"] = clustered_data["cluster_id"].astype(int)
assert len(clustered_data) == len(profiles)
assert (clustered_data.loc[X_active_scaled.index, "cluster_id"] > 0).all()
assert (clustered_data.loc[~X_interest_raw.index.isin(X_active_scaled.index), "cluster_id"] == 0).all()

cluster_sizes = clustered_data["cluster_id"].value_counts().sort_index()
cluster_summary = pd.DataFrame({
    "cluster_id": cluster_sizes.index,
    "cluster_size": cluster_sizes.values,
    "proportion": (cluster_sizes / len(clustered_data)).values,
})
interest_means = clustered_data.groupby("cluster_id")[FEATURE_COLS].mean()
interest_medians = clustered_data.groupby("cluster_id")[FEATURE_COLS].median()
interest_means.columns = [f"mean_{name}" for name in interest_means.columns]
interest_medians.columns = [f"median_{name}" for name in interest_medians.columns]
profile_stats = clustered_data.groupby("cluster_id").agg(
    mean_gradyear=("gradyear", "mean"), median_gradyear=("gradyear", "median"),
    mean_age_imputed=("age_imputed", "mean"), median_age_imputed=("age_imputed", "median"),
    mean_friends=("NumberOffriends", "mean"), median_friends=("NumberOffriends", "median"),
    mean_total_interest_counts=("total_interests", "mean"),
    median_total_interest_counts=("total_interests", "median"),
)
cluster_profile = cluster_summary.set_index("cluster_id").join(profile_stats)
cluster_profile = cluster_profile.join(interest_means).join(interest_medians).reset_index()
gender_profile = pd.crosstab(
    clustered_data["cluster_id"], clustered_data["gender_clean"], normalize="index"
).rename_axis(index="cluster_id").reset_index()

pca = PCA(n_components=2, random_state=RANDOM_STATE)
pca_coordinates = pca.fit_transform(X_active_scaled)
pca_frame = pd.DataFrame({
    "PC1": pca_coordinates[:, 0], "PC2": pca_coordinates[:, 1],
    "cluster_id": active_labels + 1,
}, index=X_active_scaled.index)
pca_plot = pca_frame.sample(
    n=min(5000, len(pca_frame)), random_state=RANDOM_STATE
)
fig, ax = plt.subplots(figsize=(10, 7))
sns.scatterplot(
    data=pca_plot, x="PC1", y="PC2", hue="cluster_id", palette="tab10",
    alpha=0.55, s=18, linewidth=0, ax=ax,
)
ax.set_title(f"PCA 2D — {best_algorithm}, k={best_cluster_count}")
ax.legend(title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "11_best_model_pca_clusters.png", dpi=150, bbox_inches="tight")
plt.show()

mean_matrix = interest_means.copy()
mean_matrix.columns = FEATURE_COLS
heatmap_values = np.log1p(mean_matrix).T
heatmap_values = heatmap_values.sub(heatmap_values.mean(axis=1), axis=0)
heatmap_values = heatmap_values.div(heatmap_values.std(axis=1).replace(0, 1), axis=0)
fig, ax = plt.subplots(figsize=(12, 14))
sns.heatmap(heatmap_values, cmap="vlag", center=0, ax=ax, cbar_kws={"label": "Z-score log1p mean"})
ax.set(title="Profil Minat per Segmen (relatif per fitur)", xlabel="Cluster ID", ylabel="Fitur minat")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "12_cluster_interest_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

top_features = X_active_raw.mean().sort_values(ascending=False).head(8).index.tolist()
bar_data = clustered_data.groupby("cluster_id")[top_features].mean().reset_index().melt(
    id_vars="cluster_id", var_name="interest", value_name="mean_count"
)
fig, ax = plt.subplots(figsize=(13, 6))
sns.barplot(data=bar_data, x="interest", y="mean_count", hue="cluster_id", ax=ax)
ax.set(title="Rata-rata hitungan fitur minat teratas per cluster", xlabel="Fitur minat", ylabel="Rata-rata hitungan mentah")
ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "13_cluster_top_interest_means.png", dpi=150, bbox_inches="tight")
plt.show()

cluster_names = {0: "No recorded interests"}
for cluster_id in sorted(set(active_labels + 1)):
    means = clustered_data.loc[clustered_data["cluster_id"] == cluster_id, FEATURE_COLS].mean()
    top_three = means.sort_values(ascending=False).head(3).index.tolist()
    cluster_names[int(cluster_id)] = "Minat tercatat: " + ", ".join(top_three)
cluster_profile["segment_description"] = cluster_profile["cluster_id"].map(cluster_names)
clustered_data["segment_description"] = clustered_data["cluster_id"].map(cluster_names)

cluster_profile.to_csv(TABLE_DIR / "cluster_profile_summary.csv", index=False)
gender_profile.to_csv(TABLE_DIR / "cluster_gender_profile.csv", index=False)
clustered_data.to_csv(TABLE_DIR / "clustered_student_profiles.csv", index=True)
model_bundle = {
    "model": best_model,
    "algorithm": best_algorithm,
    "n_clusters_active": best_cluster_count,
    "random_state": RANDOM_STATE,
    "feature_names": FEATURE_COLS,
    "feature_representation": best_feature_representation,
    "active_label_offset": 1,
    "all_zero_cluster_id": 0,
}
joblib.dump(model_bundle, MODEL_DIR / "best_clustering_model.joblib")

display(cluster_profile[[
    "cluster_id", "cluster_size", "proportion", "mean_age_imputed",
    "median_age_imputed", "mean_friends", "median_friends",
    "mean_total_interest_counts", "segment_description",
]].round(3))
display(gender_profile.round(3))
print(f"Model final: {best_algorithm}; k aktif={best_cluster_count}; profil berlabel={len(clustered_data):,}")
print("Proporsi varians PCA:", np.round(pca.explained_variance_ratio_, 4).tolist())
print("Ukuran klaster:", cluster_sizes.to_dict())

**Interpretasi segmen:** Deskripsi segmen aktif dibuat dari tiga fitur minat mentah dengan rata-rata tertinggi di segmen tersebut; deskripsi ini merangkum sinyal data, bukan label demografis atau klaim tentang individu. Klaster 0 berarti tidak ada minat yang tercatat pada 36 fitur. Gunakan ukuran/proposi, statistik usia dan teman, komposisi gender, heatmap relatif, dan PCA secara bersamaan. PCA hanya proyeksi visual dan tidak membuktikan pemisahan klaster di ruang fitur penuh.

## 8. Kesimpulan dan Rekomendasi

Tabel evaluasi dan profil di atas menjadi ringkasan hasil yang terhitung pada saat notebook dijalankan. Metrik internal tidak mengukur kegunaan bisnis secara langsung. Tinjau kestabilan segmen pada beberapa sampel/representasi, validasi nama segmen dengan konteks domain, dan dokumentasikan keterbatasan dataset sebelum mempertimbangkan deployment. Jangan gunakan segmen ini untuk keputusan berisiko tinggi terhadap siswa.